# Prepare Calendar Table

Builds the standard date dimension `healthcare_databricks.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
-- Standard calendar (date dimension): one row per day, 1940-01-01 to 2030-12-31
-- Week convention: ISO (weeks start on Monday, day_of_week 1 = Monday ... 7 = Sunday)
CREATE OR REPLACE TABLE healthcare_databricks.silver.calendar
COMMENT 'Standard calendar / date dimension from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS `date`
),
base AS (
  SELECT
    `date`,
    CAST(date_format(`date`, 'yyyyMMdd') AS INT)      AS date_key,
    year(`date`)                                      AS year,
    quarter(`date`)                                   AS quarter,
    month(`date`)                                     AS month,
    day(`date`)                                       AS day_of_month,
    weekday(`date`) + 1                               AS day_of_week,
    dayofyear(`date`)                                 AS day_of_year,
    weekofyear(`date`)                                AS week_of_year,
    extract(YEAROFWEEK FROM `date`)                   AS iso_year,
    CAST(date_trunc('WEEK', `date`) AS DATE)          AS week_start_date,
    CAST(date_trunc('MONTH', `date`) AS DATE)         AS month_start_date,
    last_day(`date`)                                  AS month_end_date,
    CAST(date_trunc('QUARTER', `date`) AS DATE)       AS quarter_start_date,
    CAST(date_trunc('YEAR', `date`) AS DATE)          AS year_start_date
  FROM dates
)
SELECT
  date_key,
  `date`,
  -- Year
  year,
  iso_year,
  CASE WHEN month <= 6 THEN 1 ELSE 2 END                         AS half_year,
  concat('H', CASE WHEN month <= 6 THEN 1 ELSE 2 END)            AS half_year_name,
  CASE WHEN (year % 4 = 0 AND year % 100 <> 0) OR year % 400 = 0
       THEN true ELSE false END                                  AS is_leap_year,
  -- Quarter
  quarter,
  concat('Q', quarter)                                           AS quarter_name,
  concat(year, '-Q', quarter)                                    AS year_quarter,
  quarter_start_date,
  date_sub(add_months(quarter_start_date, 3), 1)                 AS quarter_end_date,
  datediff(`date`, quarter_start_date) + 1                       AS day_of_quarter,
  -- Month
  month,
  date_format(`date`, 'MMMM')                                    AS month_name,
  date_format(`date`, 'MMM')                                     AS month_short_name,
  date_format(`date`, 'yyyy-MM')                                 AS year_month,
  year * 100 + month                                             AS year_month_key,
  month_start_date,
  month_end_date,
  day(month_end_date)                                            AS days_in_month,
  -- Week
  week_of_year,
  concat(iso_year, '-W', lpad(week_of_year, 2, '0'))             AS iso_year_week,
  week_start_date,
  date_add(week_start_date, 6)                                   AS week_end_date,
  CAST(ceil(day_of_month / 7.0) AS INT)                          AS week_of_month,
  -- Day
  day_of_month,
  day_of_week,
  date_format(`date`, 'EEEE')                                    AS day_name,
  date_format(`date`, 'EEE')                                     AS day_short_name,
  day_of_year,
  -- Flags
  CASE WHEN day_of_week IN (6, 7) THEN true ELSE false END       AS is_weekend,
  CASE WHEN day_of_week IN (6, 7) THEN false ELSE true END       AS is_weekday,
  `date` = month_start_date                                      AS is_month_start,
  `date` = month_end_date                                        AS is_month_end,
  `date` = quarter_start_date                                    AS is_quarter_start,
  `date` = date_sub(add_months(quarter_start_date, 3), 1)        AS is_quarter_end,
  `date` = year_start_date                                       AS is_year_start,
  (month = 12 AND day_of_month = 31)                             AS is_year_end,
  year_start_date,
  make_date(year, 12, 31)                                        AS year_end_date
FROM base
ORDER BY `date`;

In [0]:
-- Validation: expected 33,238 rows (91 years incl. 23 leap years), unique keys, correct range
SELECT
  count(*)                          AS row_count,
  count(DISTINCT date_key)          AS distinct_date_keys,
  min(`date`)                       AS min_date,
  max(`date`)                       AS max_date,
  datediff(DATE'2030-12-31', DATE'1940-01-01') + 1 AS expected_rows,
  sum(CAST(is_leap_year AND month = 2 AND day_of_month = 29 AS INT)) AS leap_days
FROM healthcare_databricks.silver.calendar;

In [0]:
SELECT * FROM healthcare_databricks.silver.calendar
WHERE `date` BETWEEN DATE'2024-12-28' AND DATE'2025-01-06'
ORDER BY `date`;